# Exercise on Joins and anti-joins: add information from other tables

In [1]:
# import libraries - solution
import pandas as pd
import numpy as np

# Set some Pandas options: maximum number of rows/columns it's going to display
#pd.set_option('display.max_rows', 1000)
#pd.set_option('display.max_columns', 100)

## Load data from clinical trial

Data comes in two different files. The file `predimed_records.csv` file contains the clinical data for each patient. The file 'predimed_location.csv' contain the information about the meaning of the location codes. Load the two dataframes, inspect them and complete the exercise below.

In [2]:
# load the dataframes
# solution
df_patients = pd.read_csv('../../data/predimed_records.csv')
df_patients

,patient-id,location-id,sex,age,smoke,bmi,waist,wth,htn,diab,hyperchol,famhist,hormo,p14,toevent,event
0,436,4,Male,58,Former,33.53,122,0.753086,No,No,Yes,No,No,10,5.374401,Yes
1,1130,4,Male,77,Current,31.05,119,0.730061,Yes,Yes,No,No,No,10,6.097194,No
2,1131,4,Female,72,Former,30.86,106,0.654321,No,Yes,No,Yes,No,8,5.946612,No
3,1132,4,Male,71,Former,27.68,118,0.694118,Yes,No,Yes,No,No,8,2.907598,Yes
4,1111,2,Female,79,Never,35.94,129,0.806250,Yes,No,Yes,No,No,9,4.761123,No
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6319,120,5,Female,66,Never,28.51,104,0.645963,Yes,No,Yes,Yes,No,8,3.550992,No
6320,118,5,Male,80,Never,23.81,109,0.589189,Yes,Yes,Yes,Yes,No,8,2.743326,No
6321,351,3,Male,57,Former,25.24,100,0.571429,Yes,No,Yes,No,NaN,7,0.479124,No
6322,499,5,Female,71,Never,32.04,98,0.653333,Yes,No,Yes,Yes,No,6,2.587269,No


There were 5 different locations where the study was conducted. For the purpose of conducting a blinded experiment, each one gave an identification number `location-id` to each participant. The data containing the information about the encoding is loaded below. It was made available after all data was collected.

In [3]:
# solution
df_locations = pd.read_csv('../../data/predimed_location.csv')

df_locations

,location-id,city
0,4,Madrid
1,1,Valencia
2,2,Barcelona
3,5,Bilbao
4,3,Malaga


### Which were the five locations?

In [4]:
# solution
df_locations['location-id'].unique()

array([4, 1, 2, 5, 3])

## Exercise

*Warm-up*

* What location-id's are present in the patient dataframe?
* For how many patients do we have clinical information? (i.e., rows in `df_patients`)
* Do all patients have an associated location information (identification number)?

In [5]:
# location-ids in patiants dataframe

df_patients['location-id'].unique()

array([4, 2, 5, 3, 1])

In [6]:
# any missing locations
df_patients['location-id'].isna().any()

np.False_

# Solutions

#### Solution 1 - O(N*M)

In [7]:
# %%timeit
# solution

patients_with_city = df_patients.copy()
patients_with_city['city_2_for_loops'] = 'n/a'

for idx, row in patients_with_city.iterrows():  # O(N)
    location = row['location-id']
    # using list comprehension
    matching_city = [r['location-id'] == location for _, r in df_locations.iterrows()]
    city = df_locations.loc[matching_city, 'city']
    if len(city) > 0:
        patients_with_city.loc[idx, 'city_2_for_loops'] = city.iloc[0]

In [8]:
# %%timeit
# solution. same as above but without list comprehension

# patients_with_city = df_patients.copy()
patients_with_city['city_2_for_loops_2'] = 'n/a'

for idx, row in patients_with_city.iterrows():  # O(N)
    location = row['location-id']
    for loc in df_locations['location-id']:
        if loc == location:
            city = df_locations['city'][df_locations['location-id'] == loc].values
    if len(city) > 0:
        patients_with_city.loc[idx, 'city_2_for_loops_2'] = city[0]

#### Solution 2 - O(N log N + M log M)
- with sorting

In [9]:
patients_with_city['city_with_sort'] = 'n/a'

sorted_patients = patients_with_city.sort_values(['location-id'])   # O(N log N)
sorted_locations = df_locations.sort_values(['location-id'])           # O(M log M)

city_2_col = sorted_patients.columns.get_loc('city_with_sort') # get the column location (number)
locations_idx = 0
patients_idx = 0

while True:    # O(N + M)
    row_locations = sorted_locations.iloc[locations_idx]
    key_locations = row_locations['location-id']

    row_patients = sorted_patients.iloc[patients_idx]
    key_patients = row_patients['location-id']

    if key_patients == key_locations:
        original_idx = sorted_patients.index[patients_idx]
        patients_with_city.iloc[original_idx, city_2_col] = row_locations['city']
        patients_idx += 1
    else:
        locations_idx += 1
        if locations_idx >= len(sorted_locations):
            break
    if patients_idx >= len(sorted_patients):
        break

In [10]:
# solution
patients_with_city['city_2_for_loops'].equals(patients_with_city['city_with_sort'])

True

#### Solution  - O(n+m)



In [11]:
# %%timeit
# solution - optimal
df_with_city = patients_with_city.merge(df_locations, 
                                        on = ['location-id'], 
                                        how = 'left')

In [12]:
patients_with_city['city_with_sort'].equals(df_with_city['city'])

True

In [15]:
# solution
# same as above but 'by hand'

patients_with_city['city_4'] = 'n/a'
# build hash table: O(M)
city_lookup = {
  (row['location-id']): row['city']
  for _, row in df_locations.iterrows()
}

# probe hash table once per row: O(N)
city_col = [
  city_lookup.get((location), np.nan)
  for location in df_patients['location-id']
]

patients_with_city['city_4'] = city_col

In [14]:
patients_with_city['city_4'].equals(df_with_city['city'])

True

In [16]:
city_lookup

{4: 'Madrid', 1: 'Valencia', 2: 'Barcelona', 5: 'Bilbao', 3: 'Malaga'}

### 4. Saving the final result in `processed_data_predimed.csv`

1. Using the `.to_csv` method of Pandas DataFrames

In [ ]:
df_with_city.to_csv('data_predimed_with_city.csv', index=None)